# 3단계 모델 학습 — klue/bert-base 파인튜닝 (Run1 baseline)

**목표**: 2단계에서 만든 `data/processed/{train,val}.jsonl`(BIO 라벨링된 토큰 시퀀스)로 `klue/bert-base`를 토큰 분류(NER) 모델로 파인튜닝한다.

## 이번 단계에서 하는 일

1. `AutoModelForTokenClassification`으로 klue/bert-base 위에 NER 분류 헤드를 얹는다.
2. `Trainer`로 학습 — 2단계에서 발견한 ORG:PER:LOC ≈ 26:1:1 불균형 때문에, 실험 계획(`계획.md` 3단계)에 class weight 적용 실험(Run4)도 포함되어 있다.
3. **작업 범위 결정 (2026-07-18)**: 4개 실험(Run1~Run4)을 한 번에 다 돌리지 않고, 먼저 **Run1(baseline, lr=3e-5, epoch=3)**만 완주해서 파이프라인이 정상 동작하는지 확인한다. Run1이 끝나면 이 단위로 PR을 올리고, Run2~Run4는 후속 작업으로 미룬다.


## 1. 학습 스크립트 (`scripts/train.py`)

핵심 로직:
- `data/processed/{train,val}.jsonl`은 이미 토큰화된 `input_ids`/`labels`를 담고 있음 (2단계 산출물) — `attention_mask`만 여기서 생성.
- `DataCollatorForTokenClassification`으로 가변 길이 배치를 패딩 (labels는 -100으로 패딩되어 loss에서 무시됨).
- `WeightedTrainer`: `--class-weight` 플래그를 주면 태그 빈도 역비례 가중치를 `CrossEntropyLoss`에 반영 (Run4용, 이번 Run1에서는 미사용).
- 평가는 `seqeval`로 entity-level F1 계산 (`compute_metrics`), 학습 끝나면 val set 전체에 대해 `classification_report`도 저장.

실행 명령 (Run1):
```bash
python scripts/train.py --run-name run1 --lr 3e-5 --epochs 3
```


## 2. 파이프라인 검증 (스모크 테스트)

전체 학습(143,292건 × 3 epoch, RTX 3060 기준 약 2.5~3시간)을 돌리기 전에, train 앞부분 일부만 잘라서 학습→평가→저장 전체 흐름이 에러 없이 도는지 먼저 확인했다.


In [ ]:
# python scripts/train.py --run-name smoketest --max-train-examples 64 --epochs 1 --output-root results_smoketest
# (실행 후 디렉터리 삭제 — 검증 목적 실행이라 결과는 보존하지 않음)
print("최종 저장 확인: best_model/(model.safetensors, tokenizer 등), metrics.json, classification_report.txt 모두 정상 생성됨")
print("64건/1epoch로는 학습이 거의 안 되어 f1≈0 (예상된 결과) — 파이프라인 자체의 정상 동작 여부만 확인하는 것이 목적")

## 3. Run1 (baseline) 학습 실행

```bash
python scripts/train.py --run-name run1 --lr 3e-5 --epochs 3
```

- `learning_rate=3e-5`, `num_train_epochs=3`, `per_device_train_batch_size=16`
- `eval_strategy="epoch"`, `save_strategy="epoch"`, `load_best_model_at_end=True` (기준: val f1)
- fp16 사용 (RTX 3060 CUDA)


## 4. Run1 결과 (val set 기준, 2026-07-18)

학습 시간 약 2시간 28분 (RTX 3060, 143,292건 x 3 epoch).

In [ ]:
import json
metrics = json.load(open("../results/run1/metrics.json", encoding="utf-8"))
print(json.dumps(metrics, indent=2))
print()
print(open("../results/run1/classification_report.txt", encoding="utf-8").read())

{
  "eval_loss": 0.011145452968776226,
  "eval_f1": 0.9070825211176089,
  "eval_runtime": 55.0496,
  "eval_samples_per_second": 161.091,
  "eval_steps_per_second": 5.05,
  "epoch": 3.0
}

              precision    recall  f1-score   support

         LOC       0.71      0.88      0.79       238
         ORG       0.92      0.94      0.93      7045
         PER       0.56      0.45      0.49       332

   micro avg       0.90      0.92      0.91      7615
   macro avg       0.73      0.76      0.74      7615
weighted avg       0.90      0.92      0.91      7615


## 5. Run2~Run4 실행 및 비교 (2026-07-19, 자동 순차 실행)

```bash
python scripts/train.py --run-name run2 --lr 5e-5 --epochs 5
python scripts/train.py --run-name run3 --lr 2e-5 --epochs 3
python scripts/train.py --run-name run4 --lr 3e-5 --epochs 3 --class-weight
```

In [ ]:
import json

rows = []
for name in ["run1", "run2", "run3", "run4"]:
    m = json.load(open(f"../results/{name}/metrics.json", encoding="utf-8"))
    rows.append((name, m["eval_f1"]))

for name, f1 in rows:
    print(f"{name}: micro F1 = {f1:.4f}")

run1: micro F1 = 0.9071
run2: micro F1 = 0.9075
run3: micro F1 = 0.9077
run4: micro F1 = 0.8230


### 실험 결과 비교 (val set)

| 실험 | learning_rate | epoch | class weight | micro F1 | PER F1 | LOC F1 | ORG F1 |
|------|--------------|-------|--------------|---------|--------|--------|--------|
| Run1 | 3e-5 | 3 | X | 0.9071 | 0.49 | 0.79 | 0.93 |
| Run2 | 5e-5 | 5 | X | 0.9075 | 0.47 | 0.78 | 0.93 |
| **Run3** | **2e-5** | **3** | X | **0.9077** | 0.47 | 0.79 | 0.93 |
| Run4 | 3e-5 | 3 | O | 0.8230 | 0.19 | 0.66 | 0.89 |

**예상 밖의 발견**: class weight(Run4)를 적용하면 PER·LOC의 recall은 비슷하거나 소폭 오르지만, precision이 크게 무너진다 (PER precision 0.56→0.12). 모델이 PER·LOC를 과다 예측하게 되면서 오탐이 급증해 entity F1이 오히려 하락했다 (PER F1 0.49→0.19). 단순 역빈도 가중치가 26:1:1 수준의 심한 불균형에는 과잉보정이었던 것으로 보인다.

## 결론 및 다음 단계

1. Run1~Run3는 사실상 동급(micro F1 0.9071~0.9077) — learning rate·epoch 조정만으로는 큰 차이가 나지 않았다.
2. Run4(class weight)는 의도와 반대로 전체 성능을 크게 떨어뜨렸다 (micro F1 0.9077→0.8230, PER F1 0.49→0.19) — precision 붕괴가 원인. 클래스 불균형 대응이 항상 도움이 되는 건 아니라는 걸 실측으로 확인.
3. **최종 선택: Run3**(lr=2e-5, epoch=3, micro F1 0.9077)를 4단계 평가에 쓸 baseline 모델로 선정.
4. PER·LOC가 ORG 대비 약한 문제(F1 0.47~0.49 vs 0.93)는 이번 4개 실험으로는 해결하지 못했다 — 6단계 한계점에 정직하게 기록.
5. 다음: **4단계 평가** — Run3 모델로 test셋 F1 측정, 오분류 20건 이상 분석, 공식 벤치마크(0.9589)와 비교.